In [2]:
!pip -q install pymupdf sentence-transformers faiss-cpu openai numpy groq

In [3]:
from sentence_transformers import SentenceTransformer

sentence = "The Little Rooftop Garden"

Model_1 = SentenceTransformer("all-MiniLM-L6-v2")
Model_2 = SentenceTransformer("all-mpnet-base-v2")

vector_1 = Model_1.encode(sentence)
vector_2 = Model_2.encode(sentence)

print(vector_1.shape)
print(vector_2.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (17 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(384,)
(768,)


In [4]:
!apt-get install -y poppler-utils
!pip -q install easyocr pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 212 kB of archives.
After this operation, 735 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 poppler-utils amd64 24.02.0-1ubuntu9.11 [212 kB]
Fetched 212 kB in 1s (310 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.11) ...
Setting up poppler-utils (24.02.0-1ubuntu9.11) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
from google.colab import files

uploaded = files.upload()

Saving The_Little_Rooftop_Garden.pdf to The_Little_Rooftop_Garden.pdf


In [6]:
uploaded = files.upload()
pdf_path = next(iter(uploaded))

Saving The_Little_Rooftop_Garden.pdf to The_Little_Rooftop_Garden (1).pdf


In [10]:

import fitz

# Open uploaded PDF
doc = fitz.open(pdf_path)

# Extract text from all pages
pdf_text = ""

for page in doc:
    pdf_text += page.get_text()

print("PDF text extracted successfully!")
print("Total characters:", len(pdf_text))
print("\nExtracted text:\n")
print(pdf_text[:2000])


PDF text extracted successfully!
Total characters: 737

Extracted text:

The Little Rooftop Garden
Mira lived in a busy apartment building where every balcony was filled with clothes,
chairs, and flower pots. One morning, she found a small packet of vegetable seeds in
her grandmother’s old sewing box. She decided to plant them in empty containers on
the rooftop. At first, her neighbours smiled at the tiny plants, but Mira watered them
carefully every day.
After a few weeks, green leaves covered the rooftop, and small tomatoes and chillies
began to grow. The neighbours started helping Mira by bringing soil, pots, and water.
Soon, the rooftop became a shared garden where everyone met in the evenings. Mira
realised that a small idea, when cared for and shared, could bring an entire community
together.



/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [11]:

# Check extracted PDF text
print(pdf_text[:500])

# Split text into chunks
chunk_size = 500
overlap = 100

chunks = []

start = 0
while start < len(pdf_text):
    end = start + chunk_size
    chunk = pdf_text[start:end]

    if chunk.strip():
        chunks.append(chunk)

    start += chunk_size - overlap

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])


The Little Rooftop Garden
Mira lived in a busy apartment building where every balcony was filled with clothes,
chairs, and flower pots. One morning, she found a small packet of vegetable seeds in
her grandmother’s old sewing box. She decided to plant them in empty containers on
the rooftop. At first, her neighbours smiled at the tiny plants, but Mira watered them
carefully every day.
After a few weeks, green leaves covered the rooftop, and small tomatoes and chillies
began to grow. The neighbour
Total chunks: 2

First chunk:
 The Little Rooftop Garden
Mira lived in a busy apartment building where every balcony was filled with clothes,
chairs, and flower pots. One morning, she found a small packet of vegetable seeds in
her grandmother’s old sewing box. She decided to plant them in empty containers on
the rooftop. At first, her neighbours smiled at the tiny plants, but Mira watered them
carefully every day.
After a few weeks, green leaves covered the rooftop, and small tomatoes and chill

In [12]:

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert chunks into embeddings
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Store embeddings
index.add(embeddings)

print("Embeddings shape:", embeddings.shape)
print("Total vectors stored:", index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (2, 384)
Total vectors stored: 2


In [13]:

from google.colab import userdata
from groq import Groq

groq_api_key = userdata.get("GROQ_KEY2")

client = Groq(api_key=groq_api_key)

print("Groq connected successfully!")


Groq connected successfully!


In [14]:

def ask_pdf(question):
    # Convert question into embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    faiss.normalize_L2(question_embedding)

    # Search relevant chunks
    distances, indices = index.search(question_embedding, 3)

    context = "\n".join(
        [chunks[i] for i in indices[0] if i < len(chunks)]
    )

    prompt = f"""
    Answer the question using only the given context.

    If the answer is not available in the context,
    say "The answer is not available in the document."

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


In [15]:

while True:
    question = input("Ask a question (type exit to stop): ")

    if question.lower() == "exit":
        print("Thank you!")
        break

    answer = ask_pdf(question)
    print("\nAnswer:", answer)
    print("-" * 50)


Ask a question (type exit to stop): Who is Mira?

Answer: Mira is a resident of the apartment building who started a rooftop garden.
--------------------------------------------------
Ask a question (type exit to stop): What did Mira find in her grandmother's sewing box?

Answer: She found a small packet of vegetable seeds.
--------------------------------------------------
Ask a question (type exit to stop): What did she grow on the rooftop?

Answer: She grew small tomatoes and chillies on the rooftop.
--------------------------------------------------
Ask a question (type exit to stop): How did the neighbours help her?

Answer: They helped by bringing soil, pots, and water.
--------------------------------------------------
Ask a question (type exit to stop): What is the lesson of the story?

Answer: The lesson of the story is that a small idea, when cared for and shared, can bring an entire community together.
--------------------------------------------------
Ask a question (type e